In [1]:
import pandas as pd
import plotly.express as px
import numpy as np
from utils import style_pts

In [2]:
df = pd.read_csv("data/team_week_points.csv")
df = df[df.manager.notnull()]
len(df)

1290

In [ ]:
#3 — Points For vs. Points Against (Luck Scatter)

# In standings_line.py or a new page
import plotly.graph_objects as go

s = pd.read_csv("data/standings_data.csv")
s = s[s.manager.notnull()]

selected_year = 2021
# Filter to final week of selected season
final = s[s['season'] == selected_year]
final = final[final['week'] == final['week'].max()]

ref = final[['points_for', 'points_against']].max().max() * 1.1
origin = final[['points_for', 'points_against']].min().min() * 0.9

fig = px.scatter(
    final, x='points_against', y='points_for',
    text='manager', color='standings_rank',
    color_continuous_scale='RdYlGn_r',
    labels={'points_for': 'Points Scored (PS)', 'points_against': 'Points Allowed (PA)'},
    title=f"Luck Chart — {selected_year}"
)
# Diagonal = perfectly even PS/PA
fig.add_shape(type='line', x0=origin, y0=origin, x1=ref, y1=ref,
              line=dict(color='gray', dash='dash'))
fig.update_traces(textposition='top center', marker=dict(size=12))
fig.update_coloraxes(showscale=False)
fig.show()
#st.plotly_chart(fig, use_container_width=True)
### should color code this by playoffs - top 4; middle 4; bottom 2

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ── shared data load ──────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    player = pd.read_csv("data/player_top.csv")
    twp    = pd.read_csv("data/team_week_points.csv")
    detail = pd.read_csv("data/player_stat_detail.csv")

    week_meta = twp[["season", "week", "week_type", "playoffs"]].drop_duplicates()
    player = player.merge(week_meta, on=["season", "week"], how="left")
    player = player[player["starting"] == 1]
    return player, detail

player, detail = load_data()

SEASONS     = sorted(player["season"].unique(), reverse=True)
WEEK_TYPES  = ["normal", "long", "short"]
MANAGERS    = sorted(player["manager_name"].dropna().unique())

# ─────────────────────────────────────────────────────────────────────────────
# #3  POINTS THRESHOLD TRACKER
# ─────────────────────────────────────────────────────────────────────────────
st.header("📊 Points Threshold Tracker")

col1, col2, col3 = st.columns(3)
season_3 = col1.selectbox("Season", SEASONS, key="s_3")
wt_3     = col2.selectbox("Week type", WEEK_TYPES, key="wt_3")

df3 = player[(player["season"] == season_3) & (player["week_type"] == wt_3)]

tab_h3, tab_p3 = st.tabs(["Hitters", "Pitchers"])

THRESHOLDS = {"B": (40, "Elite (≥40 pts)"), "P": (30, "Elite (≥30 pts)")}

for tab, ptype in [(tab_h3, "B"), (tab_p3, "P")]:
    with tab:
        default_thresh, thresh_label = THRESHOLDS[ptype]
        threshold = st.slider("Points threshold", 0, 80, default_thresh, step=5, key=f"thresh_{ptype}_3")

        counts = (df3[df3["player_type"] == ptype]
                  .groupby("manager_name")
                  .apply(lambda x: (x["pts"] >= threshold).sum())
                  .reset_index()
                  .rename(columns={"manager_name": "Manager", 0: "count"})
                  .sort_values("count", ascending=True))

        fig = px.bar(counts, x="count", y="Manager", orientation="h",
                     labels={"count": f"Weeks ≥ {threshold} pts"},
                     title=f"{'Hitter' if ptype=='B' else 'Pitcher'} weeks ≥ {threshold} pts — {season_3} ({wt_3} weeks)")
        fig.update_layout(yaxis_title="")
        st.plotly_chart(fig, use_container_width=True)

# ─────────────────────────────────────────────────────────────────────────────
# #6  TIMES IN TOP-N LEADERBOARD
# ─────────────────────────────────────────────────────────────────────────────
st.header("⭐ Times in Top-N")

col1, col2, col3 = st.columns(3)
season_6 = col1.selectbox("Season", SEASONS + ["All"], key="s_6")
wt_6     = col2.selectbox("Week type", WEEK_TYPES, key="wt_6")
top_n_6  = col3.selectbox("Top N cutoff", [3, 5, 10], key="n_6")

df6 = player[player["week_type"] == wt_6].copy()
if season_6 != "All":
    df6 = df6[df6["season"] == season_6]

# rank is already per-type per-week in the data
df6_top = df6[df6["rank"] <= top_n_6]

tab_h6, tab_p6 = st.tabs(["Hitters", "Pitchers"])

for tab, ptype in [(tab_h6, "B"), (tab_p6, "P")]:
    with tab:
        counts = (df6_top[df6_top["player_type"] == ptype]
                  .groupby(["player_name", "manager_name"])
                  .size()
                  .reset_index(name="top_n_weeks")
                  .sort_values("top_n_weeks", ascending=False)
                  .head(20))

        fig = px.bar(counts, x="top_n_weeks", y="player_name", color="manager_name",
                     orientation="h",
                     labels={"top_n_weeks": f"Weeks in top {top_n_6}", "player_name": ""},
                     title=f"{'Hitters' if ptype=='B' else 'Pitchers'} — most weeks in top {top_n_6}",
                     color_discrete_sequence=px.colors.qualitative.G10)
        fig.update_layout(yaxis={"categoryorder": "total ascending"}, legend_title="Manager")
        st.plotly_chart(fig, use_container_width=True)

# ─────────────────────────────────────────────────────────────────────────────
# #7  ROSTER QUALITY PROFILE
# ─────────────────────────────────────────────────────────────────────────────
st.header("🔬 Roster Quality Profile")

col1, col2 = st.columns(2)
season_7 = col1.selectbox("Season", SEASONS, key="s_7")
wt_7     = col2.selectbox("Week type", WEEK_TYPES, key="wt_7")

df7 = player[(player["season"] == season_7) & (player["week_type"] == wt_7)]

METRICS = {
    "B": {
        "elite_thresh":  40,   # label: "Elite weeks (≥40 pts)"
        "depth_thresh":  20,   # label: "Depth weeks (≥20 pts)"
        "weak_thresh":   10,   # label: "Weak weeks (<10 pts)"
    },
    "P": {
        "elite_thresh":  30,
        "depth_thresh":  15,
        "weak_thresh":   5,
    },
}

tab_h7, tab_p7 = st.tabs(["Hitting Roster Quality", "Pitching Roster Quality"])

for tab, ptype, label in [(tab_h7, "B", "Hitting"), (tab_p7, "P", "Pitching")]:
    with tab:
        m   = METRICS[ptype]
        sub = df7[df7["player_type"] == ptype]

        quality = (sub.groupby("manager_name")
                   .agg(
                       avg_pts       = ("pts", "mean"),
                       elite_weeks   = ("pts", lambda x: (x >= m["elite_thresh"]).sum()),
                       depth_weeks   = ("pts", lambda x: (x >= m["depth_thresh"]).sum()),
                       weak_weeks    = ("pts", lambda x: (x < m["weak_thresh"]).sum()),
                   )
                   .reset_index()
                   .rename(columns={"manager_name": "Manager"}))

        # four side-by-side bars, one metric per chart
        metric_defs = [
            ("avg_pts",     f"Avg pts/player-week",                    False, "Blues"),
            ("elite_weeks", f"Elite weeks (≥{m['elite_thresh']} pts)", False, "Greens"),
            ("depth_weeks", f"Depth weeks (≥{m['depth_thresh']} pts)", False, "YlGn"),
            ("weak_weeks",  f"Weak weeks (<{m['weak_thresh']} pts)",   True,  "Reds"),
        ]

        for metric, title, ascending, colorscale in metric_defs:
            sorted_df = quality.sort_values(metric, ascending=ascending)
            fig = px.bar(sorted_df, x="Manager", y=metric,
                         color=metric,
                         color_continuous_scale=colorscale,
                         labels={metric: title},
                         title=f"{label}: {title} — {season_7} ({wt_7} weeks)")
            fig.update_coloraxes(showscale=False)
            fig.update_layout(xaxis_title="")
            st.plotly_chart(fig, use_container_width=True)

# ─────────────────────────────────────────────────────────────────────────────
# #9  PLAYER POINTS DISTRIBUTION BY MANAGER
# ─────────────────────────────────────────────────────────────────────────────
st.header("🎻 Player Points Distribution by Manager")

col1, col2 = st.columns(2)
season_9 = col1.selectbox("Season", SEASONS, key="s_9")
wt_9     = col2.selectbox("Week type", WEEK_TYPES, key="wt_9")

df9 = player[(player["season"] == season_9) & (player["week_type"] == wt_9)]

tab_h9, tab_p9 = st.tabs(["Hitters", "Pitchers"])

for tab, ptype, label in [(tab_h9, "B", "Hitter"), (tab_p9, "P", "Pitcher")]:
    with tab:
        sub = df9[df9["player_type"] == ptype]

        # order managers by median pts descending
        order = (sub.groupby("manager_name")["pts"]
                 .median().sort_values(ascending=False).index.tolist())

        fig = px.violin(sub, x="manager_name", y="pts",
                        category_orders={"manager_name": order},
                        box=True,
                        color="manager_name",
                        color_discrete_sequence=px.colors.qualitative.G10,
                        labels={"manager_name": "Manager", "pts": "Points"},
                        title=f"{label} points distribution — {season_9} ({wt_9} weeks)")
        fig.update_layout(showlegend=False, xaxis_title="")
        st.plotly_chart(fig, use_container_width=True)